In [1]:
import warnings
warnings.simplefilter(action='ignore')

import requests, joblib, os
import numpy, pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score

In [2]:
class model_infrence:
    def __new__(self):
        self.X_train = pandas.read_csv('/kaggle/input/model-data-obd/data/X_train.csv')
        self.y_train = pandas.read_csv('/kaggle/input/model-data-obd/data/y_train.csv')
        self.X_test = pandas.read_csv('/kaggle/input/model-data-obd/data/X_test.csv')
        self.y_test = pandas.read_csv('/kaggle/input/model-data-obd/data/y_test.csv')
        self.test_c = pandas.read_csv('/kaggle/input/model-data-obd/data/X_test.csv')
        model_path = '/kaggle/input/model-data-obd/model'
        
        model_s = joblib.load(f'{model_path}/STACK.pkl')
        self.model_m = {i : joblib.load(f'{model_path}/{i}') for i in os.listdir(model_path) if i != 'STACK.pkl'}
        
        model_s_pre = model_s.predict(self.test_c)
        print(f'RMSE SCORE STACK : {100*(1-numpy.sqrt(mean_squared_error(self.y_test, model_s.predict(self.X_test))))}')
    
        results = {'STACK' : model_s_pre.reshape(-1)} | self.manual_ensemble(self)
        #print(results)
        #for i in results.keys():
            #submission = pandas.DataFrame({
                #'route_key': pandas.read_csv('/kaggle/input/redbus-decode/test_8gqdJqH (1).csv')['route_key'].values,
                #'final_seatcount': results[i],
            #})
            #submission.to_csv(f'{i}.csv', index = False)
        
    def manual_ensemble(self):
        weight, new_train_data, new_test_x_data, new_test_data, total = {}, {}, {}, {}, 0
        
        for i in self.model_m.keys():
            if i != 'KMeans':
                pre = self.model_m[i].predict(self.X_test)
                loss = numpy.sqrt(mean_squared_error(self.y_test, pre))
                weight[i] = ((1 - loss) * 0.9)#//20)-0.22)*100
                if weight[i] > 0:#loss < 1:
                    total += weight[i]
                    print(f'RMSE LOSS {i} : {loss} WEIGHT : {weight[i]}')
                
                    new_train_data[i] = (self.model_m[i].predict(self.X_train) * weight[i]).reshape(-1)
                    new_test_x_data[i] = (pre * weight[i]).reshape(-1)
                    new_test_data[i] = (self.model_m[i].predict(self.test_c) * weight[i]).reshape(-1)
        
        self.new_train_data = pandas.DataFrame(new_train_data)
        self.new_test_x_data = pandas.DataFrame(new_test_x_data)
        self.new_test_data = pandas.DataFrame(new_test_data)
        
        model = LGBMRegressor()
        model.fit(self.new_train_data, self.y_train)
        
        manual = numpy.array(sum(list(new_test_data.values()))/total)
        manual_i = numpy.array(sum(list(new_test_x_data.values()))/total)
                
        print(f'RMSE SCORE MANUAL WEIGHTED ENSEMBLE : {100*(1-numpy.sqrt(mean_squared_error(self.y_test, model.predict(self.new_test_x_data))))}')
        print(f'RMSE SCORE MANUAL WEIGHTED AVERAGE : {100*(1-numpy.sqrt(mean_squared_error(self.y_test, manual_i)))}')
        
        return {'M_META_MODEL' : model.predict(self.new_test_data).reshape(-1), 'M_AVERAGE_MODEL' :  manual.reshape(-1)}

In [3]:
model_R = model_infrence()

RMSE SCORE STACK : 26.433028217211486
RMSE LOSS SVR.pkl : 0.8812468600935208 WEIGHT : 0.10687782591583127
RMSE LOSS random_forest.pkl : 0.7625519890596877 WEIGHT : 0.21370320984628108
RMSE LOSS CatBoostRegressor.pkl : 0.7750599248146627 WEIGHT : 0.2024460676668036
RMSE LOSS LGBMRegressor.pkl : 0.9897600562890165 WEIGHT : 0.009215949339885121
RMSE LOSS XGBRegressor.pkl : 0.9430122866562883 WEIGHT : 0.0512889420093405
RMSE LOSS DecisionTreeRegressor.pkl : 0.8208511499015677 WEIGHT : 0.16123396508858906
RMSE LOSS GradientBoostingRegressor.pkl : 0.8356953125782145 WEIGHT : 0.14787421867960696
RMSE LOSS extra_trees.pkl : 0.7476211330311001 WEIGHT : 0.2271409802720099
RMSE LOSS MLPRegressor.pkl : 0.8304356685548805 WEIGHT : 0.15260789830060756
RMSE SCORE MANUAL WEIGHTED ENSEMBLE : 21.18690653173608
RMSE SCORE MANUAL WEIGHTED AVERAGE : 26.596624052538143
